# Batch Geocoding and Ambiguous Place Names

**Series:** Geocoding (notebook 3 of 3)
**Time:** about 30 minutes

**In this notebook you will:**
- Geocode many places politely with a rate limiter
- Cache results so you never geocode the same place twice
- See why names like Springfield or Paris cause errors
- Use context (state or country) to pick the right match

**You should already know:** notebooks 01a and 01b.

> **Before you start:** click **File > Save a copy in Drive** so your changes are kept.
> Then run each cell in order by clicking it and pressing **Shift + Enter**.

## Why a separate workflow for many places?

Free geocoders limit how fast you can send requests. Nominatim, for example, allows
**one request per second**. Sending hundreds of requests at once gets you blocked.

A safe batch workflow:
1. Wait between requests (rate limiting)
2. Remember every answer (a cache) so you never send the same request twice
3. Record which places failed
4. Add context to place names that are ambiguous

## Step 1: Set up a rate limited geocoder

In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(user_agent="spatialturn_batch_tutorial")

# RateLimiter waits at least 1 second between calls and retries on errors
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, max_retries=2)

## Step 2: Add a simple cache

A cache is just a dictionary: place text in, answer out. If we ask for the same place again,
we reuse the stored answer instead of sending another request.

In [ ]:
cache = {}

def lookup(place_text):
    if place_text not in cache:
        cache[place_text] = geocode(place_text, timeout=10)
    return cache[place_text]

## Step 3: Geocode a small batch

Notice that the table has a `context` column. We will see why in Step 5.

In [ ]:
places = pd.DataFrame({
    "place":   ["Paris", "Paris", "Springfield", "South Bend"],
    "context": ["",      "TX",    "IL",          "IN"],
})

def geocode_row(row):
    text = f"{row['place']}, {row['context']}" if row["context"] else row["place"]
    loc = lookup(text)
    if loc is None:
        return pd.Series({"query": text, "found": False, "address": None, "lat": None, "lon": None})
    return pd.Series({"query": text, "found": True, "address": loc.address,
                      "lat": loc.latitude, "lon": loc.longitude})

results = places.apply(geocode_row, axis=1)
results

Look at the two Paris rows. Without context, "Paris" goes to France. With "TX" it goes to Paris, Texas.

## Step 4: Check the cache

The cache now holds every answer, so asking again is instant and sends no new request.

In [ ]:
print("Cache holds", len(cache), "places")
lookup("South Bend, IN")      # answered from the cache

## Step 5: Ambiguous place names

Many names are shared: there are more than 30 places called Springfield in the U.S.
A geocoder returns its single best guess, which may not be the one you meant.

We can ask for **several candidates** (`exactly_one=False`) and look at them.

In [ ]:
candidates = geocode("Springfield, USA", exactly_one=False, limit=5, timeout=10)
for i, loc in enumerate(candidates, start=1):
    print(i, loc.address)

## Step 6: Pick the candidate that matches your context

A simple rule: prefer the candidate whose address contains the state or country you expect.

In [ ]:
def best_match(place, expected_text):
    options = geocode(place, exactly_one=False, limit=10, timeout=10) or []
    for loc in options:
        if expected_text.lower() in loc.address.lower():
            return loc
    return options[0] if options else None   # fall back to the first guess

loc = best_match("Springfield, USA", "Illinois")
print(loc.address, loc.latitude, loc.longitude)

**Best practice:** fix ambiguity in your data before geocoding. Store city, state and ZIP code
in separate columns and always send them together.

## Try it yourself

Use `best_match` to find Lafayette in Louisiana (not Indiana). Then geocode the list `['Columbus, OH', 'Columbus, IN', 'Columbus, GA']` with `lookup` and print each address.

<details>
<summary><b>Show a solution</b></summary>

```python
print(best_match("Lafayette, USA", "Louisiana").address)

for town in ["Columbus, OH", "Columbus, IN", "Columbus, GA"]:
    print(town, "->", lookup(town).address)
```

</details>

In [ ]:
# Write your answer here

## Recap

- Free geocoders have rate limits; `RateLimiter` keeps you within them.
- A cache avoids repeating requests in long batch jobs.
- Ambiguous names need context such as state, country or ZIP code.
- `exactly_one=False` returns several candidates you can compare.